In [ ]:
# Notebook 03 – Demand Aggregation & Feature Engineering

# Objective
# ---------
# Prepare the master demand dataset required for CNN demand prediction.

# This notebook performs:

# 1. Create Grid Lookup (GridID → Row, Column)
# 2. Aggregate taxi demand into 30-minute intervals
# 3. Merge weather observations
# 4. Generate temporal features
# 5. Compute historical average demand
# 6. Save the master dataset

# Outputs
# -------
# processed_data/spatial/grid_lookup.parquet

# processed_data/demand/demand_aggregation_2026_01.parquet

In [ ]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
import os
import shutil
import time # Added for small delays

mountpoint = '/content/drive'

# --- Step 1: Attempt to unmount if already mounted ---
if os.path.ismount(mountpoint):
    print(f"Detected {mountpoint} is already mounted. Attempting to unmount...")
    try:
        !fusermount -uz {mountpoint}
        print(f"Successfully unmounted {mountpoint} (if it was a fusermount). This clears any temporary files on the Colab VM's local filesystem at this location, not your actual Google Drive files.")
    except Exception as e:
        print(f"Error during fusermount unmount attempt: {e}. Proceeding with directory cleanup.")
else:
    print(f"{mountpoint} is not currently an active mount point.")

# Add a small delay to ensure unmount operations fully propagate
time.sleep(1)

# --- Step 2: Forcibly remove and recreate the mountpoint directory to ensure it's empty ---
print(f"Ensuring {mountpoint} is a fresh, empty directory before mounting...")
if os.path.exists(mountpoint):
    try:
        shutil.rmtree(mountpoint)
        print(f"Successfully removed existing {mountpoint} directory and its contents from the Colab VM's local filesystem.")
        time.sleep(0.5) # Give a moment for filesystem to register removal
    except OSError as e:
        print(f"Error removing {mountpoint}: {e}. This indicates a persistent lock or permission issue.")
        print("Attempting to create directory, but mount might still fail if directory is locked.")
else:
    print(f"Directory {mountpoint} does not exist on the Colab VM, creating it.")

# Always create the mountpoint directory to ensure it's an empty, clean target
# This ensures it's empty right before drive.mount is called
os.makedirs(mountpoint, exist_ok=True)
print(f"Created/ensured empty mountpoint directory: {mountpoint} on the Colab VM.")
time.sleep(0.5) # Another small delay

# --- Step 3: Attempt to mount Google Drive ---
print("Attempting to mount Google Drive...")
try:
    drive.mount(mountpoint, force_remount=True)
    print("Google Drive mount attempt complete.")
    if os.path.ismount(mountpoint):
        print("Google Drive successfully mounted! Your files should now be accessible via this path.")
    else:
        print("Google Drive did not mount successfully, despite no immediate exception.")
except Exception as e:
    print(f"An error occurred during drive.mount: {e}")
    print("Please check your permissions, restart the runtime, or ensure no other process is using the mountpoint.")

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras import layers, models

Detected /content/drive is already mounted. Attempting to unmount...
Successfully unmounted /content/drive (if it was a fusermount). This clears any temporary files on the Colab VM's local filesystem at this location, not your actual Google Drive files.
Ensuring /content/drive is a fresh, empty directory before mounting...
Directory /content/drive does not exist on the Colab VM, creating it.
Created/ensured empty mountpoint directory: /content/drive on the Colab VM.
Attempting to mount Google Drive...
Mounted at /content/drive
Google Drive mount attempt complete.
Google Drive successfully mounted! Your files should now be accessible via this path.


In [ ]:
# =====================================================
# PROJECT PATHS
# =====================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/RideHailingPhD")

PROCESSED_DIR = PROJECT_ROOT / "processed_data"

SPATIAL_DIR = PROCESSED_DIR / "spatial"
TRIPS_DIR = PROCESSED_DIR / "taxi_trips"
DEMAND_DIR = PROCESSED_DIR / "demand"

WEATHER_DIR = (
    PROJECT_ROOT /
    "datasets" /
    "weather" /
    "processed"
)

DEMAND_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# =====================================================
# CONFIGURATION
# =====================================================

YEAR = 2026
MONTH = 1

TIME_INTERVAL = "30min"

In [ ]:
# =====================================================
# LOAD DATASETS
# =====================================================

grid = gpd.read_file(
    SPATIAL_DIR / "grid.gpkg"
)

trips = pd.read_parquet(
    TRIPS_DIR / "tripdata_2026_01.parquet"
)

weather = pd.read_parquet(
    WEATHER_DIR / "nyc_weather_processed_2026_01.parquet"
)

In [ ]:
# =====================================================
# VALIDATE INPUTS
# =====================================================

print(f"Grid Cells     : {len(grid):,}")
print(f"Taxi Trips     : {len(trips):,}")
print(f"Weather Records: {len(weather):,}")

display(grid.head())
display(trips.head())
display(weather.head())

Grid Cells     : 1,213
Taxi Trips     : 3,699,638
Weather Records: 744


,GridID,OriginalGridID,geometry
0,0,1631,"POLYGON ((1009175.109 270121.881, 1009175.109 ..."
1,1,1682,"POLYGON ((1012175.109 270121.881, 1012175.109 ..."
2,2,1733,"POLYGON ((1015175.109 270121.881, 1015175.109 ..."
3,3,1784,"POLYGON ((1018175.109 270121.881, 1018175.109 ..."
4,4,1937,"POLYGON ((1027175.109 270121.881, 1027175.109 ..."


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,PUGridID,DOGridID
0,2026-01-01 00:54:04,2026-01-01 00:59:37,239,238,1.0,0.97,7.2,220,220
1,2026-01-01 00:34:04,2026-01-01 00:39:47,163,162,0.0,0.90,7.9,293,322
2,2026-01-01 00:57:06,2026-01-01 01:05:59,43,237,0.0,1.40,10.7,241,293
3,2026-01-01 00:15:22,2026-01-01 00:58:10,142,209,4.0,5.58,38.7,292,466
4,2026-01-01 00:27:13,2026-01-01 00:40:43,88,144,0.0,2.16,13.5,498,467


,Datetime,Temperature,WindSpeed,WeatherCode
0,2026-01-01 00:00:00,-0.0,17.0,1.0
1,2026-01-01 01:00:00,1.0,17.0,3.0
2,2026-01-01 02:00:00,1.0,17.0,3.0
3,2026-01-01 03:00:00,1.0,15.0,3.0
4,2026-01-01 04:00:00,1.0,19.0,3.0


In [ ]:
# # =====================================================
# # STAGE 1 : CREATE GRID LOOKUP
# # =====================================================

# ## Objective

# The CNN requires every GridID to have a fixed location in a 2D image.

# This stage computes:

# - Row Index
# - Column Index
# - Grid Centroid Coordinates

# The lookup generated here will be reused throughout the remaining notebooks.

# ### Input

# - grid.gpkg

# ### Output

# processed_data/spatial/grid_lookup.parquet

In [ ]:
# =====================================================
# CREATE GRID LOOKUP
# =====================================================

# Create a copy to preserve the original dataframe
grid_lookup = grid.copy()

# -----------------------------------------------------
# Compute centroid coordinates
# -----------------------------------------------------

grid_lookup["CentroidX"] = grid_lookup.geometry.centroid.x
grid_lookup["CentroidY"] = grid_lookup.geometry.centroid.y

# -----------------------------------------------------
# Determine unique grid coordinates
# -----------------------------------------------------

unique_x = np.sort(grid_lookup["CentroidX"].unique())
unique_y = np.sort(grid_lookup["CentroidY"].unique())[::-1]

# -----------------------------------------------------
# Create lookup dictionaries
# -----------------------------------------------------

column_lookup = {
    x: idx
    for idx, x in enumerate(unique_x)
}

row_lookup = {
    y: idx
    for idx, y in enumerate(unique_y)
}

# -----------------------------------------------------
# Assign Row and Column
# -----------------------------------------------------

grid_lookup["Row"] = grid_lookup["CentroidY"].map(row_lookup).astype(int)

grid_lookup["Column"] = grid_lookup["CentroidX"].map(column_lookup).astype(int)

# Keep only required columns
grid_lookup = grid_lookup[
    [
        "GridID",
        "OriginalGridID",
        "Row",
        "Column",
        "CentroidX",
        "CentroidY",
        "geometry"
    ]
]

In [ ]:
# =====================================================
# VALIDATE GRID LOOKUP
# =====================================================

num_rows = grid_lookup["Row"].nunique()

num_cols = grid_lookup["Column"].nunique()

image_cells = num_rows * num_cols

empty_cells = image_cells - len(grid_lookup)

print("=" * 60)

print("GRID LOOKUP SUMMARY")

print("=" * 60)

print(f"Valid Grid Cells : {len(grid_lookup):,}")

print(f"Grid Rows        : {num_rows}")

print(f"Grid Columns     : {num_cols}")

print(f"Image Size       : {num_rows} x {num_cols}")

print(f"Image Cells      : {image_cells:,}")

print(f"Empty Cells      : {empty_cells:,}")

print("=" * 60)

# -----------------------------------------------------
# Data integrity checks
# -----------------------------------------------------

assert grid_lookup["GridID"].is_unique, \
    "Duplicate GridIDs found."

assert not grid_lookup[
    ["Row", "Column"]
].duplicated().any(), \
    "Duplicate Row/Column locations found."

print()

print("Grid lookup validation passed.")

display(grid_lookup.head())

GRID LOOKUP SUMMARY
Valid Grid Cells : 1,213
Grid Rows        : 51
Grid Columns     : 52
Image Size       : 51 x 52
Image Cells      : 2,652
Empty Cells      : 1,439

Grid lookup validation passed.


,GridID,OriginalGridID,Row,Column,CentroidX,CentroidY,geometry
0,0,1631,0,31,1.007675e+06,271621.881254,"POLYGON ((1009175.109 270121.881, 1009175.109 ..."
1,1,1682,0,32,1.010675e+06,271621.881254,"POLYGON ((1012175.109 270121.881, 1012175.109 ..."
2,2,1733,0,33,1.013675e+06,271621.881254,"POLYGON ((1015175.109 270121.881, 1015175.109 ..."
3,3,1784,0,34,1.016675e+06,271621.881254,"POLYGON ((1018175.109 270121.881, 1018175.109 ..."
4,4,1937,0,37,1.025675e+06,271621.881254,"POLYGON ((1027175.109 270121.881, 1027175.109 ..."


In [ ]:
# =====================================================
# SAVE GRID LOOKUP
# =====================================================

grid_lookup_file = (
    SPATIAL_DIR /
    "grid_lookup.parquet"
)

grid_lookup.to_parquet(
    grid_lookup_file,
    index=False
)

print()

print(f"Saved : {grid_lookup_file}")


Saved : /content/drive/MyDrive/RideHailingPhD/processed_data/spatial/grid_lookup.parquet


In [ ]:
# =====================================================
# VALIDATE GRID MASK
# =====================================================

grid_mask = np.zeros((num_rows, num_cols), dtype=np.uint8)

for _, row in grid_lookup.iterrows():
    grid_mask[row["Row"], row["Column"]] = 1

print(f"Mask Shape       : {grid_mask.shape}")
print(f"Valid Cells      : {grid_mask.sum()}")
print(f"Invalid Cells    : {(grid_mask == 0).sum()}")

assert grid_mask.sum() == len(grid_lookup)

display(pd.DataFrame(grid_mask).head())

Mask Shape       : (51, 52)
Valid Cells      : 1213
Invalid Cells    : 1439


,0,1,2,3,4,5,6,7,8,9,...,42,43,44,45,46,47,48,49,50,51
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0


In [ ]:
mask_file = SPATIAL_DIR / "grid_mask.npy"

np.save(mask_file, grid_mask)

print(f"Saved : {mask_file}")

Saved : /content/drive/MyDrive/RideHailingPhD/processed_data/spatial/grid_mask.npy


In [ ]:
# # =====================================================
# # STAGE 2 : DEMAND AGGREGATION
# # =====================================================

# ## Objective

# Aggregate taxi pickup demand into fixed 30-minute intervals.

# Each row in the output represents one spatial grid during one
# 30-minute period.

# A complete spatio-temporal table is created so that every
# time slot contains every valid grid cell. Missing demand
# values are filled with zero.

# ### Input

# - tripdata_2026_01.parquet
# - grid_lookup.parquet

# ### Output

# Demand table

In [ ]:
# =====================================================
# CREATE 30-MINUTE TIME SLOTS
# =====================================================

# Create a working copy
trip_data = trips.copy()

# Ensure datetime format
trip_data["tpep_pickup_datetime"] = pd.to_datetime(
    trip_data["tpep_pickup_datetime"]
)

# Floor timestamps to the nearest 30 minutes
trip_data["TimeSlot"] = (
    trip_data["tpep_pickup_datetime"]
        .dt.floor(TIME_INTERVAL)
)

print(f"Unique Time Slots : {trip_data['TimeSlot'].nunique():,}")

display(
    trip_data[
        [
            "tpep_pickup_datetime",
            "TimeSlot",
            "PUGridID"
        ]
    ].head()
)

Unique Time Slots : 1,490


,tpep_pickup_datetime,TimeSlot,PUGridID
0,2026-01-01 00:54:04,2026-01-01 00:30:00,220
1,2026-01-01 00:34:04,2026-01-01 00:30:00,293
2,2026-01-01 00:57:06,2026-01-01 00:30:00,241
3,2026-01-01 00:15:22,2026-01-01 00:00:00,292
4,2026-01-01 00:27:13,2026-01-01 00:00:00,498


In [ ]:
# =====================================================
# KEEP ONLY JANUARY 2026
# =====================================================

trip_data = trip_data[
    (
        trip_data["TimeSlot"] >= "2026-01-01 00:00:00"
    )
    &
    (
        trip_data["TimeSlot"] < "2026-02-01 00:00:00"
    )
].copy()

print(f"Trips Remaining : {len(trip_data):,}")

print()

print(
    trip_data["TimeSlot"].min(),
    "→",
    trip_data["TimeSlot"].max()
)

Trips Remaining : 3,699,633

2026-01-01 00:00:00 → 2026-01-31 23:30:00


In [ ]:
# =====================================================
# AGGREGATE DEMAND
# =====================================================

demand_table = (
    trip_data
        .groupby(
            [
                "TimeSlot",
                "PUGridID"
            ],
            observed=True
        )
        .size()
        .reset_index(name="Demand")
)

print(f"Aggregated Rows : {len(demand_table):,}")

display(demand_table.head())

Aggregated Rows : 368,413


,TimeSlot,PUGridID,Demand
0,2026-01-01,9,1
1,2026-01-01,32,1
2,2026-01-01,33,1
3,2026-01-01,41,1
4,2026-01-01,51,1


In [ ]:
# =====================================================
# CREATE COMPLETE SPACE-TIME GRID
# =====================================================

# All 30-minute intervals in January
time_slots = pd.date_range(
    start=trip_data["TimeSlot"].min(),
    end=trip_data["TimeSlot"].max(),
    freq=TIME_INTERVAL
)

# Every valid grid
grid_ids = grid_lookup["GridID"].sort_values().unique()

# Cartesian product
space_time = pd.MultiIndex.from_product(
    [time_slots, grid_ids],
    names=["TimeSlot", "GridID"]
).to_frame(index=False)

print(f"Expected Rows : {len(space_time):,}")

display(space_time.head())

Expected Rows : 1,804,944


,TimeSlot,GridID
0,2026-01-01,0
1,2026-01-01,1
2,2026-01-01,2
3,2026-01-01,3
4,2026-01-01,4


In [ ]:
# =====================================================
# MERGE DEMAND
# =====================================================

master_demand = (
    space_time.merge(
        demand_table,
        left_on=[
            "TimeSlot",
            "GridID"
        ],
        right_on=[
            "TimeSlot",
            "PUGridID"
        ],
        how="left"
    )
)

# Remove duplicate key
master_demand.drop(
    columns=["PUGridID"],
    inplace=True
)

# Missing demand means zero pickups
master_demand["Demand"] = (
    master_demand["Demand"]
        .fillna(0)
        .astype(np.int16)
)

master_demand = master_demand.merge(
    grid_lookup[
        ["GridID", "Row", "Column"]
    ],
    on="GridID",
    how="left"
)

master_demand = (
    master_demand
    .sort_values(
        [
            "TimeSlot",
            "GridID"
        ]
    )
    .reset_index(drop=True)
)

assert master_demand["Row"].isna().sum() == 0
assert master_demand["Column"].isna().sum() == 0

print(f"Master Rows : {len(master_demand):,}")

display(master_demand.head())

Master Rows : 1,804,944


,TimeSlot,GridID,Demand,Row,Column
0,2026-01-01,0,0,0,31
1,2026-01-01,1,0,0,32
2,2026-01-01,2,0,0,33
3,2026-01-01,3,0,0,34
4,2026-01-01,4,0,0,37


In [ ]:
# =====================================================
# VALIDATE DEMAND TABLE
# =====================================================

print("=" * 60)

print("DEMAND TABLE SUMMARY")

print("=" * 60)

print(f"Time Slots      : {master_demand['TimeSlot'].nunique():,}")

print(f"Grid Cells      : {master_demand['GridID'].nunique():,}")

print(f"Total Rows      : {len(master_demand):,}")

print(f"Total Demand    : {master_demand['Demand'].sum():,}")

print("=" * 60)

# -----------------------------------------------------
# Validation
# -----------------------------------------------------

assert (
    master_demand["Demand"].sum()
    ==
    len(trip_data)
)

expected_rows = (
    master_demand["TimeSlot"].nunique()
    *
    master_demand["GridID"].nunique()
)

assert len(master_demand) == expected_rows

print()

print("Demand aggregation validation passed.")

DEMAND TABLE SUMMARY
Time Slots      : 1,488
Grid Cells      : 1,213
Total Rows      : 1,804,944
Total Demand    : 3,699,633

Demand aggregation validation passed.


In [ ]:
# =====================================================
# VALIDATE TIME SLOT RANGE
# =====================================================

print("First Time Slot :", master_demand["TimeSlot"].min())
print("Last Time Slot  :", master_demand["TimeSlot"].max())

print()
print("Unique Time Slots :", master_demand["TimeSlot"].nunique())

print()
print("First 5 Time Slots")
display(
    pd.DataFrame(
        master_demand["TimeSlot"]
        .drop_duplicates()
        .sort_values()
        .head()
    )
)

print()
print("Last 5 Time Slots")
display(
    pd.DataFrame(
        master_demand["TimeSlot"]
        .drop_duplicates()
        .sort_values()
        .tail()
    )
)

First Time Slot : 2026-01-01 00:00:00
Last Time Slot  : 2026-01-31 23:30:00

Unique Time Slots : 1488

First 5 Time Slots


,TimeSlot
0,2026-01-01 00:00:00
1213,2026-01-01 00:30:00
2426,2026-01-01 01:00:00
3639,2026-01-01 01:30:00
4852,2026-01-01 02:00:00



Last 5 Time Slots


,TimeSlot
1798879,2026-01-31 21:30:00
1800092,2026-01-31 22:00:00
1801305,2026-01-31 22:30:00
1802518,2026-01-31 23:00:00
1803731,2026-01-31 23:30:00


In [ ]:
# # =====================================================
# # STAGE 3 : WEATHER INTEGRATION
# # =====================================================

# ## Objective

# Merge hourly weather observations with every 30-minute
# demand interval.

# Each hourly weather observation is assigned to the two
# corresponding 30-minute demand intervals.

# ### Input

# - Demand Table
# - Processed Weather

# ### Output

# Master Demand Table with Weather Features

In [ ]:
# =====================================================
# PREPARE WEATHER DATA
# =====================================================

weather_table = weather.copy()

weather_table["Datetime"] = pd.to_datetime(
    weather_table["Datetime"]
)

weather_table["WeatherHour"] = (
    weather_table["Datetime"]
        .dt.floor("h")
)

display(weather_table.head())

,Datetime,Temperature,WindSpeed,WeatherCode,WeatherHour
0,2026-01-01 00:00:00,-0.0,17.0,1.0,2026-01-01 00:00:00
1,2026-01-01 01:00:00,1.0,17.0,3.0,2026-01-01 01:00:00
2,2026-01-01 02:00:00,1.0,17.0,3.0,2026-01-01 02:00:00
3,2026-01-01 03:00:00,1.0,15.0,3.0,2026-01-01 03:00:00
4,2026-01-01 04:00:00,1.0,19.0,3.0,2026-01-01 04:00:00


In [ ]:
# =====================================================
# PREPARE DEMAND TABLE
# =====================================================

master_demand["WeatherHour"] = (
    master_demand["TimeSlot"]
        .dt.floor("h")
)

display(
    master_demand[
        [
            "TimeSlot",
            "WeatherHour"
        ]
    ].head()
)

,TimeSlot,WeatherHour
0,2026-01-01,2026-01-01
1,2026-01-01,2026-01-01
2,2026-01-01,2026-01-01
3,2026-01-01,2026-01-01
4,2026-01-01,2026-01-01


In [ ]:
# =====================================================
# MERGE WEATHER
# =====================================================

master_demand = (
    master_demand.merge(
        weather_table[
            [
                "WeatherHour",
                "Temperature",
                "WindSpeed",
                "WeatherCode"
            ]
        ],
        on="WeatherHour",
        how="left"
    )
)

# WeatherHour was only needed for joining
master_demand.drop(columns=["WeatherHour"], inplace=True)

In [ ]:
# =====================================================
# VALIDATE WEATHER MERGE
# =====================================================

print("=" * 60)

print("WEATHER MERGE SUMMARY")

print("=" * 60)

print(f"Rows : {len(master_demand):,}")

print()

print("Missing Values")

print(master_demand[
    [
        "Temperature",
        "WindSpeed",
        "WeatherCode"
    ]
].isna().sum())

print()

assert master_demand["Temperature"].isna().sum() == 0
assert master_demand["WindSpeed"].isna().sum() == 0
assert master_demand["WeatherCode"].isna().sum() == 0

print("Weather merge validation passed.")

display(master_demand.head())

WEATHER MERGE SUMMARY
Rows : 1,804,944

Missing Values
Temperature    0
WindSpeed      0
WeatherCode    0
dtype: int64

Weather merge validation passed.


,TimeSlot,GridID,Demand,Row,Column,Temperature,WindSpeed,WeatherCode
0,2026-01-01,0,0,0,31,-0.0,17.0,1.0
1,2026-01-01,1,0,0,32,-0.0,17.0,1.0
2,2026-01-01,2,0,0,33,-0.0,17.0,1.0
3,2026-01-01,3,0,0,34,-0.0,17.0,1.0
4,2026-01-01,4,0,0,37,-0.0,17.0,1.0


In [ ]:
# =====================================================
# CREATE TEMPORAL FEATURES
# =====================================================

master_demand["DayOfWeek"] = (
    master_demand["TimeSlot"].dt.dayofweek.astype(np.int8)
)

master_demand["Hour"] = (
    master_demand["TimeSlot"].dt.hour.astype(np.int8)
)

master_demand["Minute"] = (
    master_demand["TimeSlot"].dt.minute.astype(np.int8)
)

# 30-minute period within the day (0-47)
master_demand["Period"] = (
    master_demand["Hour"] * 2 +
    (master_demand["Minute"] // 30)
).astype(np.int8)

In [ ]:
# =====================================================
# CREATE TIME INDEX
# =====================================================

unique_slots = (
    master_demand["TimeSlot"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

time_lookup = pd.DataFrame({
    "TimeSlot": unique_slots,
    "TimeIndex": np.arange(len(unique_slots), dtype=np.int16)
})

master_demand = master_demand.merge(
    time_lookup,
    on="TimeSlot",
    how="left"
)

In [ ]:
# =====================================================
# VALIDATE TEMPORAL FEATURES
# =====================================================

print("=" * 60)
print("TEMPORAL FEATURE SUMMARY")
print("=" * 60)

print("Unique Days      :", master_demand["DayOfWeek"].nunique())
print("Unique Hours     :", master_demand["Hour"].nunique())
print("Unique Periods   :", master_demand["Period"].nunique())
print("Unique TimeIndex :", master_demand["TimeIndex"].nunique())

print()

print("DayOfWeek :", sorted(master_demand["DayOfWeek"].unique()))
print("Hour      :", sorted(master_demand["Hour"].unique()))
print("Period    :", sorted(master_demand["Period"].unique())[:10], "...")

assert master_demand["DayOfWeek"].between(0, 6).all()
assert master_demand["Hour"].between(0, 23).all()
assert master_demand["Period"].between(0, 47).all()
assert master_demand["TimeIndex"].nunique() == 1488

print("\nTemporal feature validation passed.")

display(
    master_demand[
        [
            "TimeSlot",
            "TimeIndex",
            "DayOfWeek",
            "Hour",
            "Period"
        ]
    ].head()
)

TEMPORAL FEATURE SUMMARY
Unique Days      : 7
Unique Hours     : 24
Unique Periods   : 48
Unique TimeIndex : 1488

DayOfWeek : [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6)]
Hour      : [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9), np.int8(10), np.int8(11), np.int8(12), np.int8(13), np.int8(14), np.int8(15), np.int8(16), np.int8(17), np.int8(18), np.int8(19), np.int8(20), np.int8(21), np.int8(22), np.int8(23)]
Period    : [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9)] ...

Temporal feature validation passed.


,TimeSlot,TimeIndex,DayOfWeek,Hour,Period
0,2026-01-01,0,3,0,0
1,2026-01-01,0,3,0,0
2,2026-01-01,0,3,0,0
3,2026-01-01,0,3,0,0
4,2026-01-01,0,3,0,0


In [ ]:
# =====================================================
# COMPUTE HISTORICAL AVERAGE DEMAND
# =====================================================

group_columns = [
    "GridID",
    "DayOfWeek",
    "Period"
]

# Sort chronologically within each group
master_demand = (
    master_demand
    .sort_values(
        group_columns + ["TimeIndex"]
    )
    .reset_index(drop=True)
)

# Previous cumulative demand
previous_sum = (
    master_demand
    .groupby(group_columns)["Demand"]
    .cumsum()
    - master_demand["Demand"]
)

# Number of previous observations
previous_count = (
    master_demand
    .groupby(group_columns)
    .cumcount()
)

# Historical average using only previous occurrences
master_demand["HistoricalAverage"] = (
    previous_sum /
    previous_count.replace(0, np.nan)
)

# First occurrence has no history
master_demand["HistoricalAverage"] = (
    master_demand["HistoricalAverage"]
    .fillna(0)
    .astype(np.float32)
)

In [ ]:
# =====================================================
# VALIDATE HISTORICAL AVERAGE
# =====================================================

print("=" * 70)
print("HISTORICAL DEMAND SUMMARY")
print("=" * 70)

print(f"Rows                     : {len(master_demand):,}")
print(f"Average Historical Value : {master_demand['HistoricalAverage'].mean():.3f}")
print(f"Maximum Historical Value : {master_demand['HistoricalAverage'].max():.3f}")
print(f"Zero Historical Values   : {(master_demand['HistoricalAverage'] == 0).sum():,}")

print()

display(
    master_demand[
        [
            "GridID",
            "TimeSlot",
            "Demand",
            "HistoricalAverage"
        ]
    ].head(20)
)

HISTORICAL DEMAND SUMMARY
Rows                     : 1,804,944
Average Historical Value : 1.514
Maximum Historical Value : 369.000
Zero Historical Values   : 1,420,537



,GridID,TimeSlot,Demand,HistoricalAverage
0,0,2026-01-05 00:00:00,0,0.0
1,0,2026-01-12 00:00:00,0,0.0
2,0,2026-01-19 00:00:00,0,0.0
3,0,2026-01-26 00:00:00,0,0.0
4,0,2026-01-05 00:30:00,0,0.0
5,0,2026-01-12 00:30:00,0,0.0
6,0,2026-01-19 00:30:00,0,0.0
7,0,2026-01-26 00:30:00,0,0.0
8,0,2026-01-05 01:00:00,0,0.0
9,0,2026-01-12 01:00:00,0,0.0


In [ ]:
# =====================================================
# TOP 10 BUSIEST GRIDS
# =====================================================

busy_grids = (
    master_demand
    .groupby("GridID")["Demand"]
    .sum()
    .sort_values(ascending=False)
)

print(busy_grids.head(10))

GridID
293    174387
292    150652
321    136674
320    132959
379    112397
350    110866
268    109625
351     92131
409     92059
294     84872
Name: Demand, dtype: int64


In [ ]:
grid = busy_grids.index[0]

display(
    master_demand[
        master_demand["GridID"] == grid
    ][
        [
            "TimeSlot",
            "Demand",
            "HistoricalAverage"
        ]
    ].head(25)
)

,TimeSlot,Demand,HistoricalAverage
435984,2026-01-05 00:00:00,15,0.000000
435985,2026-01-12 00:00:00,24,15.000000
435986,2026-01-19 00:00:00,36,19.500000
435987,2026-01-26 00:00:00,11,25.000000
435988,2026-01-05 00:30:00,6,0.000000
435989,2026-01-12 00:30:00,17,6.000000
435990,2026-01-19 00:30:00,15,11.500000
435991,2026-01-26 00:30:00,8,12.666667
435992,2026-01-05 01:00:00,10,0.000000
435993,2026-01-12 01:00:00,8,10.000000


In [ ]:
master_demand = (
    master_demand
    .sort_values(
        [
            "TimeIndex",
            "GridID"
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
grid_id = 293

display(
    master_demand[
        master_demand["GridID"] == grid_id
    ][
        [
            "TimeSlot",
            "Demand",
            "HistoricalAverage"
        ]
    ].head(40)
)

,TimeSlot,Demand,HistoricalAverage
293,2026-01-01 00:00:00,154,0.0
1506,2026-01-01 00:30:00,202,0.0
2719,2026-01-01 01:00:00,195,0.0
3932,2026-01-01 01:30:00,166,0.0
5145,2026-01-01 02:00:00,113,0.0
6358,2026-01-01 02:30:00,100,0.0
7571,2026-01-01 03:00:00,60,0.0
8784,2026-01-01 03:30:00,45,0.0
9997,2026-01-01 04:00:00,34,0.0
11210,2026-01-01 04:30:00,18,0.0


In [ ]:
grid_id = 293

check = (
    master_demand[
        (master_demand["GridID"] == grid_id) &
        (master_demand["Period"] == 0)          # 00:00–00:30
    ][
        [
            "TimeSlot",
            "Demand",
            "HistoricalAverage"
        ]
    ]
)

display(check)

,TimeSlot,Demand,HistoricalAverage
293,2026-01-01,154,0.000000
58517,2026-01-02,24,0.000000
116741,2026-01-03,37,0.000000
174965,2026-01-04,51,0.000000
233189,2026-01-05,15,0.000000
291413,2026-01-06,25,0.000000
349637,2026-01-07,22,0.000000
407861,2026-01-08,38,154.000000
466085,2026-01-09,59,24.000000
524309,2026-01-10,82,37.000000


In [ ]:
master_demand.rename(
    columns={
        "HistoricalAverage": "HistoricalDemand"
    },
    inplace=True
)

In [ ]:
# =====================================================
# FINAL DATASET SUMMARY
# =====================================================

print("=" * 70)
print("FINAL MASTER DATASET")
print("=" * 70)

print(f"Rows               : {len(master_demand):,}")
print(f"Columns            : {master_demand.shape[1]}")
print(f"Time Slots         : {master_demand['TimeIndex'].nunique():,}")
print(f"Grid Cells         : {master_demand['GridID'].nunique():,}")
print(f"Total Demand       : {master_demand['Demand'].sum():,}")

print()

print("Columns")
print("-" * 70)

for column in master_demand.columns:
    print(column)

print()

display(master_demand.head())

display(master_demand.tail())

FINAL MASTER DATASET
Rows               : 1,804,944
Columns            : 14
Time Slots         : 1,488
Grid Cells         : 1,213
Total Demand       : 3,699,633

Columns
----------------------------------------------------------------------
TimeSlot
GridID
Demand
Row
Column
Temperature
WindSpeed
WeatherCode
DayOfWeek
Hour
Minute
Period
TimeIndex
HistoricalDemand



,TimeSlot,GridID,Demand,Row,Column,Temperature,WindSpeed,WeatherCode,DayOfWeek,Hour,Minute,Period,TimeIndex,HistoricalDemand
0,2026-01-01,0,0,0,31,-0.0,17.0,1.0,3,0,0,0,0,0.0
1,2026-01-01,1,0,0,32,-0.0,17.0,1.0,3,0,0,0,0,0.0
2,2026-01-01,2,0,0,33,-0.0,17.0,1.0,3,0,0,0,0,0.0
3,2026-01-01,3,0,0,34,-0.0,17.0,1.0,3,0,0,0,0,0.0
4,2026-01-01,4,0,0,37,-0.0,17.0,1.0,3,0,0,0,0,0.0


,TimeSlot,GridID,Demand,Row,Column,Temperature,WindSpeed,WeatherCode,DayOfWeek,Hour,Minute,Period,TimeIndex,HistoricalDemand
1804939,2026-01-31 23:30:00,1208,0,49,5,-4.0,7.0,3.0,5,23,30,47,1487,0.0
1804940,2026-01-31 23:30:00,1209,0,50,0,-4.0,7.0,3.0,5,23,30,47,1487,0.0
1804941,2026-01-31 23:30:00,1210,0,50,1,-4.0,7.0,3.0,5,23,30,47,1487,0.0
1804942,2026-01-31 23:30:00,1211,0,50,2,-4.0,7.0,3.0,5,23,30,47,1487,0.0
1804943,2026-01-31 23:30:00,1212,0,50,3,-4.0,7.0,3.0,5,23,30,47,1487,0.0


In [ ]:
# =====================================================
# SAVE MASTER DATASET
# =====================================================

DEMAND_DIR = PROCESSED_DIR / "demand"

DEMAND_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    DEMAND_DIR /
    "demand_master_2026_01.parquet"
)

master_demand.to_parquet(
    output_file,
    index=False
)

print(f"Saved : {output_file}")

Saved : /content/drive/MyDrive/RideHailingPhD/processed_data/demand/demand_master_2026_01.parquet
